# 03 — Core A/B Analysis

**The question:** does sending a marketing email cause incremental purchases and revenue —
and if so, which email should we send?

**Pre-registered design** (from `02_experiment_design.py`, committed *before* looking at
outcomes, so the goalposts can't move):

- **Primary metric:** 2-week conversion. **Secondary:** visit, spend.
- **alpha = 0.05** two-sided; two treatments vs one control ⇒ **Holm correction** on every
  metric family.
- **Decision rule:** ship an email iff its Holm-adjusted conversion p-value < 0.05 **and**
  the 95% CI on incremental spend excludes zero.
- Experiment validity was established in `01_eda.ipynb` (SRM + balance both pass).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utils import (two_proportion_ztest, welch_ttest, bootstrap_diff_ci,
                   holm_correction)

sns.set_theme(style="whitegrid", context="notebook")

df = pd.read_csv("../data/raw/hillstrom.csv")

ctrl = df[df["segment"] == "No E-Mail"]
mens = df[df["segment"] == "Mens E-Mail"]
womens = df[df["segment"] == "Womens E-Mail"]
ARMS = {"Mens E-Mail": mens, "Womens E-Mail": womens}
print({name: len(arm) for name, arm in ARMS.items()},
      "| control:", len(ctrl))

## 1. Primary metric: conversion

Binary outcome + huge n ⇒ **two-proportion z-test** (see `utils.py` for why the test uses a
pooled standard error while the CI uses an unpooled one).

We test each treatment against control, then apply **Holm's correction**: with two tests,
each at 5% false-positive risk, the chance of at least one fluke is ~10%; Holm shrinks the
family-wise error back to 5% without giving up as much power as plain Bonferroni.

In [ ]:
def test_binary_metric(metric):
    """Both treatments vs control on a 0/1 column, Holm-corrected."""
    rows = []
    for name, arm in ARMS.items():
        r = two_proportion_ztest(int(arm[metric].sum()), len(arm),
                                 int(ctrl[metric].sum()), len(ctrl))
        rows.append({"arm": name, "treatment rate": r.p_treat,
                     "control rate": r.p_ctrl, "abs lift": r.abs_lift,
                     "rel lift": r.rel_lift, "ci_low": r.ci_low,
                     "ci_high": r.ci_high, "p raw": r.p_value})
    out = pd.DataFrame(rows).set_index("arm")
    # Correct within the metric family (2 tests per metric).
    out["p Holm"], out["significant"] = holm_correction(out["p raw"])
    return out

conv = test_binary_metric("conversion")
conv.style.format({"treatment rate": "{:.3%}", "control rate": "{:.3%}",
                   "abs lift": "{:+.3%}", "rel lift": "{:+.1%}",
                   "ci_low": "{:+.3%}", "ci_high": "{:+.3%}",
                   "p raw": "{:.2e}", "p Holm": "{:.2e}"})

**Reading the table:** the *absolute* lift (percentage points) is what converts to dollars;
the *relative* lift is what sounds impressive in a headline. Report both, always — "+119%"
and "+0.68pp" describe the same effect, and a stakeholder who hears only one of them has
been half-informed.

## 2. Secondary metric: visit

Same machinery. Visits are ~15x more common than conversions, so this test has far more
statistical resolution — useful as a mechanism check: *did the email at least pull people
to the site?*

In [ ]:
visit = test_binary_metric("visit")
visit.style.format({"treatment rate": "{:.2%}", "control rate": "{:.2%}",
                    "abs lift": "{:+.2%}", "rel lift": "{:+.1%}",
                    "ci_low": "{:+.2%}", "ci_high": "{:+.2%}",
                    "p raw": "{:.2e}", "p Holm": "{:.2e}"})

## 3. Secondary metric: spend — the money question

Spend per customer is **zero-inflated** (99%+ zeros) and **heavily right-skewed** among
buyers. Two-pronged approach:

1. **Welch's t-test** — valid for comparing *means* of large samples via the CLT, and robust
   to the unequal variances that revenue data always has.
2. **Bootstrap CI** — 10,000 resamples, zero distributional assumptions. If it agrees with
   Welch (it will), the parametric shortcut was safe; if it disagreed, we'd trust the
   bootstrap.

In [ ]:
spend_rows = []
boot_results = {}
for name, arm in ARMS.items():
    w = welch_ttest(arm["spend"], ctrl["spend"])
    b = bootstrap_diff_ci(arm["spend"], ctrl["spend"], n_boot=10_000)
    boot_results[name] = b
    spend_rows.append({
        "arm": name,
        "mean spend (treat)": w["mean_treat"],
        "mean spend (ctrl)": w["mean_ctrl"],
        "diff $/customer": w["diff"],
        "Welch CI": f"[{w['ci_low']:+.3f}, {w['ci_high']:+.3f}]",
        "Bootstrap CI": f"[{b['ci_low']:+.3f}, {b['ci_high']:+.3f}]",
        "p raw": w["p_value"],
    })

spend = pd.DataFrame(spend_rows).set_index("arm")
spend["p Holm"], spend["significant"] = holm_correction(spend["p raw"])
spend.style.format({"mean spend (treat)": "${:.3f}",
                    "mean spend (ctrl)": "${:.3f}",
                    "diff $/customer": "${:+.3f}",
                    "p raw": "{:.2e}", "p Holm": "{:.2e}"})

In [ ]:
# Visual proof that Welch and the bootstrap agree: the bootstrap sampling
# distribution of the spend difference, with both CIs overlaid.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)

for ax, (name, b) in zip(axes, boot_results.items()):
    ax.hist(b["boot_diffs"], bins=60, alpha=0.85)
    ax.axvline(0, color="crimson", ls="--", lw=1.5, label="zero effect")
    ax.axvline(b["ci_low"], color="black", ls=":", lw=1.5)
    ax.axvline(b["ci_high"], color="black", ls=":", lw=1.5,
               label="bootstrap 95% CI")
    ax.set_title(f"{name} - ctrl: bootstrap of $\Delta$ mean spend")
    ax.set_xlabel("$ per customer")
    ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig("../assets/bootstrap_spend.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. All effects on one chart (forest plot)

The single most information-dense way to present A/B results: every effect with its 95% CI.
Anything whose interval clears zero is a real effect; the interval width is the honesty bar
that a bare "significant!" headline hides.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

panels = [("conversion", conv, "abs lift (pp)", 100),
          ("visit", visit, "abs lift (pp)", 100)]

for ax, (metric, table, xlabel, scale) in zip(axes[:2], panels):
    for i, (arm_name, row) in enumerate(table.iterrows()):
        color = "#4878cf" if "Mens" in arm_name else "#e07a9b"
        ax.errorbar(row["abs lift"] * scale, i,
                    xerr=[[(row["abs lift"] - row["ci_low"]) * scale],
                          [(row["ci_high"] - row["abs lift"]) * scale]],
                    fmt="o", capsize=5, color=color, markersize=9)
    ax.axvline(0, color="crimson", ls="--", lw=1.2)
    ax.set_yticks(range(len(table)))
    ax.set_yticklabels(table.index)
    ax.set_xlabel(xlabel)
    ax.set_title(f"{metric} lift vs control")

ax = axes[2]
for i, (arm_name, row) in enumerate(spend.iterrows()):
    b = boot_results[arm_name]
    color = "#4878cf" if "Mens" in arm_name else "#e07a9b"
    ax.errorbar(row["diff $/customer"], i,
                xerr=[[row["diff $/customer"] - b["ci_low"]],
                      [b["ci_high"] - row["diff $/customer"]]],
                fmt="o", capsize=5, color=color, markersize=9)
ax.axvline(0, color="crimson", ls="--", lw=1.2)
ax.set_yticks(range(len(spend)))
ax.set_yticklabels(spend.index)
ax.set_xlabel("$ / customer (bootstrap CI)")
ax.set_title("spend lift vs control")

fig.suptitle("Treatment effects with 95% confidence intervals", y=1.04,
             fontweight="bold")
fig.tight_layout()
fig.savefig("../assets/forest_plot.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. From lift to dollars — the number the C-suite actually asked for

The experiment measures **incremental spend per emailed customer** (treatment mean − control
mean). Control already captures what customers would have spent anyway, so this difference
is pure causal impact — no baseline subtraction gymnastics needed.

Scaling assumption (stated, not hidden): effects transfer proportionally to a larger
customer file *of the same composition*. A national retailer emails millions, not 21k.

In [ ]:
mens_lift_per_customer = spend.loc["Mens E-Mail", "diff $/customer"]
mens_ci_low = boot_results["Mens E-Mail"]["ci_low"]
mens_ci_high = boot_results["Mens E-Mail"]["ci_high"]

# Sensitivity across file sizes and email frequency: a range, not one number,
# because the honest projection carries its uncertainty with it.
print(f"Incremental spend per emailed customer (Mens E-Mail): "
      f"${mens_lift_per_customer:.3f}  "
      f"(95% CI ${mens_ci_low:.3f} to ${mens_ci_high:.3f})\n")

print(f"{'file size':>12} | {'sends/yr':>8} | {'point estimate':>16} | "
      f"{'95% CI range':>28}")
print("-" * 75)
for file_size in [1_000_000, 5_000_000, 10_000_000]:
    for sends in [12, 26]:  # monthly vs biweekly campaign cadence
        pt = file_size * sends * mens_lift_per_customer
        lo = file_size * sends * mens_ci_low
        hi = file_size * sends * mens_ci_high
        print(f"{file_size:>12,} | {sends:>8} | ${pt:>14,.0f} | "
              f"${lo:>11,.0f} - ${hi:>11,.0f}")

**Caveats that keep this honest** (also in `docs/03_stakeholder_faq.md`):

- The per-send effect will **not** stay constant at 26 sends/year — list fatigue is real.
  Treat the biweekly rows as an upper bound; the honest claim is the *monthly* row.
- The effect was measured over a 2-week window; longer-horizon cannibalization (customers
  buying now instead of later) is not visible in this design.
- Scaling assumes the larger file has the same composition as the test population.

## 6. Verdict against the pre-registered decision rule

| Rule | Mens E-Mail | Womens E-Mail |
|---|---|---|
| Holm-adjusted conversion p < 0.05 | YES | YES |
| Spend CI excludes zero | YES | NO (interval spans 0) |
| **Decision** | **SHIP** | **DO NOT SHIP (yet)** |

The Men's email is the unambiguous winner: it roughly **doubles conversion** and produces
measurable incremental revenue. The Women's email lifts conversion but its revenue effect is
statistically indistinguishable from zero — under the pre-registered rule, it doesn't ship.
That is the point of committing to the rule in advance: "significant on one metric" is not
the same as "makes money".

Next: `04_segmentation.ipynb` asks *who* drives the Men's email effect — because "send to
everyone" is almost never the revenue-maximizing policy.